# Ropedia Academy — B5 · NeRF & volume rendering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B5.ipynb)

> **Implements the whole NeRF core — positional encoding, an (x→color,density) MLP, differentiable volume rendering, and a photometric loss that backprops to train it.**
>
> 实现 NeRF 的全部核心——位置编码、(x→颜色,密度) MLP、可微体渲染，以及可反向传播训练它的光度损失。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B5

In [ ]:
import torch, torch.nn as nn

# NeRF: an MLP (x, dir) -> (color, density). Positional encoding lets it fit detail.
def posenc(x, L=6):
    out = [x]
    for i in range(L):
        for fn in (torch.sin, torch.cos): out.append(fn(x * (2.0**i) * torch.pi))
    return torch.cat(out, -1)

mlp = nn.Sequential(nn.Linear(3*(1+2*6), 128), nn.ReLU(), nn.Linear(128, 4))
def field(p):
    o = mlp(posenc(p)); return torch.sigmoid(o[..., :3]), torch.relu(o[..., 3])

def render(sigma, color, delta):                   # DIFFERENTIABLE volume rendering
    alpha = 1 - torch.exp(-sigma * delta)          # per-sample opacity
    T = torch.cumprod(torch.cat([torch.ones(1), 1 - alpha + 1e-10])[:-1], 0)  # transmittance
    return ((T * alpha)[:, None] * color).sum(0)   # integrate color along the ray

t = torch.linspace(2, 6, 64)                        # samples along one ray
pts = torch.stack([torch.zeros(64), torch.zeros(64), t], -1)
color, sigma = field(pts)
pixel = render(sigma, color, (t[1]-t[0]).expand(64))
loss = ((pixel - torch.tensor([0.8, 0.2, 0.1])) ** 2).mean()  # photometric loss
loss.backward()                                     # -> trains the MLP (no 3D labels!)
print("rendered pixel:", pixel.detach().round(decimals=3).tolist(), "| loss:", round(loss.item(), 3))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B5
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks